# Fine-tuning Chronos on solar data (`pv_kw`)

In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use one GPU so runs are comparable
!pip install -q transformers datasets accelerate chronos-forecasting wandb 

In [2]:
import random, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments
from chronos import ChronosPipeline

SEED = 42
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - turn on a GPU in the Settings panel")

GPU: Tesla T4


## Step 1: Load the data
We load the microgrid dataset directly from huggingface. We'll use the `pv_kw` column, which is hourly solar generation.

In [3]:
URL = ("https://huggingface.co/datasets/Tristanchou/"
       "Preference-Conditioned-Heterogeneous-MARL-Microgrid/resolve/main/microgrid_opsd_multiyear.csv")

# Parse timestamp to convert it to computer-readable date-time objects
df = pd.read_csv(URL, parse_dates=["timestamp"])

print(f"The dataset shape: {df.shape}")
print( df.head())


HTTPError: HTTP Error 500: Internal Server Error

In [ ]:
# confirm that there are no missing values
df.isnull().sum()

In [ ]:

pv = df.set_index("timestamp")["pv_kw"].astype("float32")

print("Gap check (should be only 1 hour):", pv.index.to_series().diff().dropna().value_counts().to_dict())

In [ ]:
# let's see what the target looks like
print(df["pv_kw"].describe())

In [ ]:
# Set timestamp as the index
pv = df.set_index("timestamp")["pv_kw"]["2016-01-01":]   # about 4.8 years
# we index out pv_kw because Chronos forecasts one stream of numbers at a time

print(pv.index.to_series().diff().value_counts().head())  # the interval should be all 1 hour

In [ ]:
pv

## Step 2: Split chronologically for time-series data
The model trains on the earliest set, and we tune settings on the validation set.

In [ ]:
PRED_LEN = 24                       # prediction length, forecast 24 hours ahead
VAL_DAYS, TEST_DAYS = 30, 30

series = pv.values
series   # all the hours 

In [ ]:
# To get the row where hour for test set begins: total number of hours - (number of test days * 24 hours)

test_start = len(series) - TEST_DAYS * 24
val_start  = test_start - VAL_DAYS * 24

print(f"train: {len(series[:val_start])} hours")
print(f"validation: {len(series[val_start:test_start])} hours")
print(f"test: {len(series[test_start:])} hours")

## Step 3: Prepating the data with sliding window  
Here, we use the sliding window technique, a method used to process sequential data by maintaining a subset of elements (a "window") that moves through the data incrementally. 

### Why this method?
We cannot feed the entire solar data into the model at once. So the function takes a fixed-size frame (a window) and slides it across the dataset 24 hours at a time. It then stamps out lots of mini training examples.  The function sets the History (Context) and the Future (Horizon) 


- `make_windows` inputs:  
        - values: the full array of hourly PV readings.  
        - first_target and last_target: the range of positions where the answers must fall. For example, to build training windows, you'd pass the start and end of the training period.  
        - context_length: how many past hours the model sees as input.  
        - stride: how many hours to move forward between windows.


- `make_windows` cuts the pv series into pairs of (context, target).

- `tokenize_windows` coverts the pairs into tokens, and returns them in dictionary format

In [ ]:
def make_windows(values, first_target, last_target, context_length, stride):
    contexts = []
    targets = []
    start = max(first_target, context_length)      # chooses the bigger value because we need enough history before the first target
    
    while start + PRED_LEN <= last_target:        #start + PRED_LEN is the value just after the last target
        contexts.append(values[start - context_length : start])     # use previous values to form the context(histroy)
        targets.append(values[start : start + PRED_LEN])            # use latter values to form the target (future/horizon)
        start = start + stride
    return np.array(contexts, dtype="float32"), np.array(targets, dtype="float32")

In [ ]:
def tokenize_windows(pipe, contexts, targets):
    tokenizer = pipe.tokenizer      # pipe is the chronos pipeline we imported at the beginning
    tokenizer.config.prediction_length = PRED_LEN      # We set the tokenizer rediction length to 24hours so it knows what to expect 
    
    examples = []
    for i in range(len(contexts)):
        context = torch.tensor(contexts[i]).unsqueeze(0)     # add a batch dimension of 1 as that's what the model expects:  2D in form of (batch_size, time_steps)
        target = torch.tensor(targets[i]).unsqueeze(0)
        
        # here, we do the mean scaling (divides the values by a scale factor which is often the average size of the context values) 
        # and quantizing(binning) where it places each scaled number into one out of a fixed set of bins
        tokens, mask, scale = tokenizer.context_input_transform(context)
        labels, labels_mask = tokenizer.label_input_transform(target, scale)

#         The names that the model's forward function expects:
            # input_ids and attention_mask: the history tokens and their mask.
            # labels and decoder_attention_mask: the answer tokens and their mask.
        examples.append({
            "input_ids": tokens.squeeze(0),                  # remove the batch dimension again
            "attention_mask": mask.squeeze(0),
            "labels": labels.squeeze(0),
            "decoder_attention_mask": labels_mask.squeeze(0),
        })
    return examples

## Evaluating the model

In [ ]:
# # Sets up the test data

def make_eval_data(values, eval_start, n_days, context_length):
    contexts = []
    actuals = []
    yesterdays = []
    
    for day in range(n_days):
        t = eval_start + day * PRED_LEN                      # the hour this day's forecast begins
        contexts.append(values[t - context_length : t])
        actuals.append(values[t : t + PRED_LEN])
        yesterdays.append(values[t - PRED_LEN : t])          # the 24 hours just before
        
    return (np.array(contexts, dtype="float32"),         # For each day, it returns the historical solar data before that day,
            np.array(actuals, dtype="float32"),          # the value for that day 
            np.array(yesterdays, dtype="float32"))       # 24 hours just before the forecast

Below, we compare the model against a **naive persistence baseline**
A naive persistence baseline is a simple time-series forecasting benchmark that predicts a future value will equal the most recently observed value

In [ ]:
# Compares the model to a naive baseline

def get_scores(pipe, values, eval_start, n_days, context_length):
    contexts, actuals, yesterdays = make_eval_data(values, eval_start, n_days, context_length)
    forecasts = pipe.predict(torch.tensor(contexts), prediction_length=PRED_LEN, num_samples=20)
    forecasts = forecasts.float().cpu().numpy()              # shape: (days, 20 guesses, 24 hours)
    median = np.median(forecasts, axis=1)                    # middle of the 20 guesses
    model_mae = np.mean(np.abs(median - actuals))
    naive_mae = np.mean(np.abs(yesterdays - actuals))
    return float(model_mae), float(naive_mae)

In [ ]:
# runs the fine-tuning

def train_model(pipe, train_examples, val_examples, learning_rate, epochs, batch_size,
                seed, use_wandb, run_name):
    args = TrainingArguments(
        output_dir="/kaggle/working/trainer_tmp",
        learning_rate=learning_rate,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        logging_steps=50,
        eval_strategy="epoch",              # check the validation loss after every epoch
        save_strategy="no",
        seed=seed,
        report_to="wandb" if use_wandb else "none",
        run_name=run_name,
    )
    trainer = Trainer(model=pipe.model.model, args=args,
                      train_dataset=train_examples, eval_dataset=val_examples)
    trainer.train()                         # updates the model inside pipe, in place
    return trainer

In [ ]:
def run_experiment(model_name="amazon/chronos-t5-mini", context_length=168, learning_rate=1e-4,
                   epochs=1, batch_size=16, stride=6, seed=42, use_wandb=False,
                   final_test=False, save_dir=None, run_name=None):
    set_seed(seed)
    results = {}

    if use_wandb and wandb.run is None:                      # 0. start tracking
        wandb.init(project=WANDB_PROJECT, name=run_name,
                   config={"model_name": model_name, "context_length": context_length,
                           "learning_rate": learning_rate, "epochs": epochs,
                           "batch_size": batch_size, "stride": stride, "seed": seed})

    pipe = ChronosPipeline.from_pretrained(model_name, device_map="cuda", torch_dtype=torch.float32)  # 1

    val_model, val_naive = get_scores(pipe, series, val_start, VAL_DAYS, context_length)   # 2. BEFORE
    results["val_mae_zero_shot"] = val_model
    results["val_mae_naive"] = val_naive
    if final_test:
        test_model, test_naive = get_scores(pipe, series, test_start, TEST_DAYS, context_length)
        results["test_mae_zero_shot"] = test_model
        results["test_mae_naive"] = test_naive

    train_x, train_y = make_windows(series, 0, val_start, context_length, stride)          # 3. prepare
    val_x, val_y = make_windows(series, val_start, test_start, context_length, PRED_LEN)
    train_examples = tokenize_windows(pipe, train_x, train_y)
    val_examples = tokenize_windows(pipe, val_x, val_y)
    results["n_train_windows"] = len(train_examples)

    start_time = time.time()                                                               # 4. train
    trainer = train_model(pipe, train_examples, val_examples, learning_rate, epochs,
                          batch_size, seed, use_wandb, run_name)
    results["train_minutes"] = round((time.time() - start_time) / 60, 1)

    val_model, _ = get_scores(pipe, series, val_start, VAL_DAYS, context_length)           # 5. AFTER
    results["val_mae_finetuned"] = val_model
    if final_test:
        test_model, _ = get_scores(pipe, series, test_start, TEST_DAYS, context_length)
        results["test_mae_finetuned"] = test_model

    if save_dir is not None:                                                               # 6. wrap up
        trainer.save_model(save_dir)
    if use_wandb:
        wandb.log(results)
        wandb.finish()
    del pipe, trainer
    torch.cuda.empty_cache()
    return results

In [ ]:
res = run_experiment()
for k, v in res.items():
    print(f"{k:22s} {v}")

In [ ]:
# Grid search with a CSV log

import csv, os

def log_result(settings, results, path="/kaggle/working/tuning_log.csv"):
    row = dict(settings)
    row.update(results)
    new_file = not os.path.exists(path)
    with open(path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if new_file:
            writer.writeheader()              # write the column names once
        writer.writerow(row)

learning_rates = [3e-5, 1e-4, 3e-4, 1e-3]
epoch_options = [1, 2, 3]

for lr in learning_rates:
    for ep in epoch_options:
        settings = {"learning_rate": lr, "epochs": ep}
        results = run_experiment(learning_rate=lr, epochs=ep)
        log_result(settings, results)
        print(settings, "->", round(results["val_mae_finetuned"], 3))